# Module 2 — Vector search (lessons 02–06)

Course: [02-vector-search](https://github.com/DataTalksClub/llm-zoomcamp/tree/main/02-vector-search)

Keyword search (module 1) matches **tokens**. Vector search matches **meaning**: an embedding model maps each
text to a point in a 384-dimensional space, and texts that mean similar things land close together.
This notebook goes from a single embedding to a vector-search RAG.

## Lesson 02 — Embeddings

`all-MiniLM-L6-v2`: 384 dimensions, fast on CPU, outputs **unit-length** vectors.
For unit vectors, dot product = cosine similarity, so `a.dot(b)` is all we need.

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1, v2, dv = model.encode(q1), model.encode(q2), model.encode(d)

print("shape:", v1.shape, "| norm:", round(float(np.linalg.norm(v1)), 4))
print(f"join-question vs doc:   {v1.dot(dv):.2f}")
print(f"docker-question vs doc: {v2.dot(dv):.2f}")

shape: (384,) | norm: 1.0
join-question vs doc:   0.32
docker-question vs doc: 0.02


The registration document shares almost no words with "Can I still join…", yet scores far higher than the
Docker question. Keyword search can't see that relationship. Vector search can.

## Lesson 03 — Embedding the FAQ dataset

Question and answer are embedded together, so a query can match either. Encoding in batches of 50 gives a progress
bar and keeps memory flat.

In [3]:
from tqdm.auto import tqdm
from ingest import load_faq_data

documents = load_faq_data()
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

batch_size = 50
vectors = []
for i in tqdm(range(0, len(texts), batch_size)):
    vectors.extend(model.encode(texts[i:i + batch_size]))

X = np.array(vectors)
X.shape

  0%|          | 0/30 [00:00<?, ?it/s]

(1489, 384)

The FAQ is a living dataset: at the time of this run it has more documents than the 1,208 in the lesson video, so indices and scores differ slightly.

## Lesson 04 — Vector search with numpy

`X.dot(v)` is one matrix-vector product: row `i` of the result is the cosine similarity between document `i` and the
query. That's brute-force (exact) nearest-neighbour search.

In [4]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

scores = X.dot(v_query)
top5 = np.argsort(-scores)[:5]   # negate -> argsort gives descending order

for idx in top5:
    print(f"{scores[idx]:.3f}  [{documents[idx]['course']}] {documents[idx]['question']}")

0.763  [data-engineering-zoomcamp] Course: Can I still join the course after the start date?
0.758  [mlops-zoomcamp] Course - Can I still join the course after the start date?
0.719  [machine-learning-zoomcamp] The course has already started. Can I still join it?
0.654  [llm-zoomcamp] I just discovered the course. Can I still join?
0.560  [data-engineering-zoomcamp] Course - Can I follow the course after it finishes?


## Lesson 05 — minsearch `VectorSearch`

Same brute-force dot product, wrapped with the same `fit`/`search`/`filter_dict` API as the text `Index`.

In [5]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

query = "I just discovered the course. Can I still join it?"
results = vindex.search(model.encode(query), filter_dict={"course": "llm-zoomcamp"}, num_results=5)
[r["question"] for r in results]

['I just discovered the course. Can I still join?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'Can I run the course locally instead of Codespaces?',
 'Does the course certificate show the number of course hours?',
 'OpenAI: Do I have to subscribe and pay for Open AI API for this course?']

### Extra — keyword vs vector on a paraphrase

The course claims vector search "handles rephrased questions better". A query that shares few words with the FAQ
entry "I just discovered the course. Can I still join?" tests that claim directly.

In [6]:
from ingest import build_index

text_index = build_index(documents)
paraphrase = "the program has already begun, can I still sign up?"

keyword_hits = text_index.search(
    paraphrase, filter_dict={"course": "llm-zoomcamp"},
    boost_dict={"question": 3.0, "section": 0.5}, num_results=5,
)
vector_hits = vindex.search(model.encode(paraphrase), filter_dict={"course": "llm-zoomcamp"}, num_results=5)

print("KEYWORD")
for r in keyword_hits:
    print("  -", r["question"])
print("VECTOR")
for r in vector_hits:
    print("  -", r["question"])

KEYWORD
  - The homework submission form is still open even though the deadline has passed — can I still submit?
  - I just discovered the course. Can I still join?
  - I’ve already submitted my project. Why can’t I review any projects?
  - I missed the first homework - can I still get a certificate?
  - My dataset is copyrighted and I can't publish it. Can I still use it for the project?
VECTOR
  - I just discovered the course. Can I still join?
  - Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
  - I passed the project. When will I get my certificate?
  - Can I run the course locally instead of Codespaces?
  - The homework submission form is still open even though the deadline has passed — can I still submit?


**Observed:** vector search ranks the right entry #1 and keyword search ranks it #2, so keyword still finds it
because "can I still" is distinctive. The difference is in the *rest* of the list. Keyword fills it with
"can I still submit / use / get…" entries (shared words, unrelated topics). Vector returns registration and
course-logistics entries (related meaning). The gain is real but modest here. That's why the course says
to adopt vector search only once evaluation (module 4) shows keyword search missing results.

## Lesson 06 — RAG with vector search

RAG is modular, so only `search` changes. The query must be embedded before searching.

In [7]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from rag_helper import RAGBase

load_dotenv()
groq_client = OpenAI(api_key=os.getenv("GROQ_API_KEY"), base_url="https://api.groq.com/openai/v1")


class RAGVector(RAGBase):
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        return self.index.search(
            self.embedder.encode(query),
            num_results=num_results,
            filter_dict={"course": self.course},
        )


vector_assistant = RAGVector(embedder=model, index=vindex, llm_client=groq_client)
print(vector_assistant.rag(paraphrase))

Yes—you can still join even though the program has already started. Just sign up, and if you want a certificate you’ll need to submit your project before the submission window closes.
